# Compare local experiments
Compare matching split and subset sizes. Full DEV and smoke subsets must not be interpreted as the same benchmark.

In [1]:
from pathlib import Path
import sys, json
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p/'asr/config.py').is_file()), None)
if PROJECT is None:
    PROJECT = next((p/'ASR' for p in [here, *here.parents] if (p/'ASR/asr/config.py').is_file()), None)
assert PROJECT is not None, 'Open notebook from the project workspace'
PYTHON = PROJECT.parent/'.venv-asr/Scripts/python.exe'
DATA = PROJECT/'data'
RAW_DATA = PROJECT.parent/'dataset/ASR'
assert PYTHON.is_file(), 'Run ASR/local/setup.ps1 first'
sys.path.insert(0, str(PROJECT/'local'))
from notebook_process import run_cli as _run_cli
def run_cli(arguments):
    return _run_cli(PROJECT, PYTHON, arguments)
RUNNER = PROJECT/'scripts/local_run.py'
print('Project:', PROJECT)
print('Python:', PYTHON)
print('Ready. Run the specific task cell; model jobs do not start in this setup cell.')


Project: D:\1.FPT_FALL26\DPL - Deep Learning\ASR
Python: D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe
Ready. Run the specific task cell; model jobs do not start in this setup cell.


In [2]:
import pandas as pd
rows = []
for path in (PROJECT/'local/runs').glob('*/*/eval/*/metrics.json'):
    m = json.loads(path.read_text(encoding='utf-8'))
    rel = path.relative_to(PROJECT/'local/runs')
    if rel.parts[1].startswith('validation'): continue
    rows.append({'experiment':rel.parts[0], 'run':rel.parts[1], 'split_or_subset':rel.parts[3], 'n':m['overall']['n_utts'], 'WER (%)':100*m['overall']['wer'], 'CER (%)':100*m['overall']['cer'], 'checkpoint':m['checkpoint']})
comparison = pd.DataFrame(rows)
display(comparison)
if rows:
    comparison.to_csv(PROJECT/'local/runs/comparison.csv',index=False)

,experiment,run,split_or_subset,n,WER (%),CER (%),checkpoint
0,B0_phowhisper_small_zeroshot,dev_local_v1,dev,5168,24.726670,19.251341,vinai/PhoWhisper-small
1,B0_phowhisper_small_zeroshot,smoke_eval,dev_sub2,8,15.596330,8.735868,vinai/PhoWhisper-small
2,B1_whisper_small_zeroshot,dev_local_v1,dev,5168,58.751631,46.354632,openai/whisper-small
3,B1_whisper_small_zeroshot,smoke_eval,dev_sub2,8,30.275229,18.602261,openai/whisper-small
4,V1_phowhisper_small_lora,dev_local_v1,dev,5168,18.478575,14.364676,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...
5,V1_phowhisper_small_lora,smoke_local_v1,dev_sub2,8,10.091743,5.035971,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...


In [3]:
resources = []
for path in (PROJECT/'local/runs').glob('*/*/resources.jsonl'):
    entries = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if entries:
        resources.append({'run':str(path.parent.relative_to(PROJECT)), 'max_total_gpu_GB':max(e['gpu_total_used_gb'] for e in entries), 'max_host_RSS_GB':max(e['host_rss_gb'] for e in entries)})
display(pd.DataFrame(resources))

,run,max_total_gpu_GB,max_host_RSS_GB
0,local\runs\B0_phowhisper_small_zeroshot\dev_lo...,1.747452,1.599611
1,local\runs\B0_phowhisper_small_zeroshot\smoke_...,1.743258,1.752441
2,local\runs\B0_phowhisper_small_zeroshot\valida...,1.642594,0.732975
3,local\runs\B0_phowhisper_small_zeroshot\valida...,1.741160,1.592238
4,local\runs\B1_whisper_small_zeroshot\dev_local_v1,1.751646,1.690960
5,local\runs\B1_whisper_small_zeroshot\smoke_eval,1.741160,1.637736
6,local\runs\B1_whisper_small_zeroshot\validatio...,1.120403,0.722838
7,local\runs\V1_phowhisper_small_lora\dev_local_v1,1.785201,1.696158
8,local\runs\V1_phowhisper_small_lora\local_v1,3.123184,4.068790
9,local\runs\V1_phowhisper_small_lora\smoke_loca...,1.785201,1.552151
